In [112]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from algorithms.methods import *
from utils.read_data import *
from utils.postprocess import *
from utils.read_data import *
from utils.math_utils import *
from simulation import Simulation, add_awgn, denoising_process
from orchestrator import ExperimentOrchestrator
import re

np.random.seed(10)
pd.set_option('display.precision', 3)

In [10]:
X,y = read_s_dataset(4)
X_part, y_part = random_data_partition(X,y,5)

# Effect of denoising

In [3]:
np.random.seed(1) 
X_client = X_part[0]
centers = kmeans(X_client, 15)
# X_noisy = add_awgn(centers, 20, eps=0.)
X_denoised1, _ = denoising_process(centers, 1, 30, 0., scheme="rep", solver="ols")
X_denoised2, _ = denoising_process(centers, 1, 20, 0., scheme="rep", solver="ols")
X_denoised3, _ = denoising_process(centers, 1, 10, 0., scheme="rep", solver="ols")


# Row 1 Data (Example parameters)
denoised_row1 = [
    denoising_process(centers, 1, 30, 0., scheme="rep", solver="ols")[0],
    denoising_process(centers, 1, 20, 0., scheme="rep", solver="ols")[0],
    denoising_process(centers, 1, 10, 0., scheme="rep", solver="ols")[0]
]

# Row 2 Data (Change a parameter, e.g., different solver or scheme)
denoised_row2 = [
    denoising_process(centers, 4, 30, 0., scheme="rep", solver="ols")[0],
    denoising_process(centers, 4, 20, 0., scheme="rep", solver="ols")[0],
    denoising_process(centers, 4, 10, 0., scheme="rep", solver="ols")[0]
]

# --- Plotting ---
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=('SNR 30dB', 'SNR 20dB', 'SNR 10dB', 
                                    'SNR 30dB (r=4)', 'SNR 20dB (r=4)', 'SNR 10dB (r=4)'),
                                    vertical_spacing=0.12)

colors = {'client': '#636EFA', 'sent': '#EF553B', 'received': '#00CC96'}

def add_bundle(fig, row, col, rx_data, show_lgd):
    # Client Data
    fig.add_trace(go.Scatter(x=X_client[:, 0], y=X_client[:, 1], mode='markers', 
                             name='Client Data', marker_color=colors['client'],
                             legendgroup='g1', showlegend=show_lgd), row=row, col=col)
    # Sent Centers
    fig.add_trace(go.Scatter(x=centers[:, 0], y=centers[:, 1], mode='markers', 
                             name='Sent Centroids', marker_color=colors['sent'],
                             legendgroup='g2', showlegend=show_lgd), row=row, col=col)
    # Received/Denoised Centers
    fig.add_trace(go.Scatter(x=rx_data[:, 0], y=rx_data[:, 1], mode='markers', 
                             name='Recovered Centroids', marker_color=colors['received'],
                             legendgroup='g3', showlegend=show_lgd), row=row, col=col)

# Fill Row 1
for i, data in enumerate(denoised_row1):
    # Only show legend on the very first subplot (Row 1, Col 1)
    is_first = (i == 0)
    add_bundle(fig, 1, i+1, data, is_first)

# Fill Row 2
for i, data in enumerate(denoised_row2):
    # legend is always False here because Row 1 already created it
    add_bundle(fig, 2, i+1, data, False)

fig.update_layout(
    height=500, 
    # title_text='Comparison of Denoising Parameters',
    margin=dict(t=30, b=30, l=20, r=20),
    showlegend=True,
    template='plotly_white',
)

fig.show()

In [122]:
n_run = 20
r_list = [1, 2, 4, 8, 16, 32, 64]
noise = [10, 20, 30]
np.random.seed(1)
results = []  # will hold dicts with r, ns, mse_mean, mse_std

for r in r_list:
    for ns in noise:
        mse_trials = []
        for _ in range(n_run):
            X_denoised, _ = denoising_process(centers, r, ns, 0.0, scheme="rep", solver="ols")
            mse = np.sum((X_denoised - centers) ** 2)   # fixed variable name
            mse_trials.append(mse)
        
        mse_mean = np.mean(mse_trials)
        mse_std = np.std(mse_trials, ddof=1)  # sample standard deviation
        
        results.append({
            'r': r,
            'noise': ns,
            'mse_mean': mse_mean,
            'mse_std': mse_std
        })
results_df = pd.DataFrame(results)
import plotly.graph_objects as go

# Group data by noise level
noise_levels = sorted(set(res['noise'] for res in results))
r_values = sorted(set(res['r'] for res in results))

fig = go.Figure()

for ns in noise_levels:
    # Extract data for this noise level, sorted by r
    data_ns = [res for res in results if res['noise'] == ns]
    data_ns.sort(key=lambda x: x['r'])  # ensure x-axis order
    
    x = [res['r'] for res in data_ns]
    y = [res['mse_mean'] for res in data_ns]
    y_std = [res['mse_std'] for res in data_ns]
    
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        name=f'Noise = {ns}',
        error_y=dict(
            type='data',
            array=y_std,
            visible=True,
            thickness=1.5,
            width=6
        ),
        line=dict(width=2),
        marker=dict(size=8)
    ))

# Customize layout
fig.update_layout(
    # title='Denoising effect: MSE vs. r',
    xaxis_title='r (log)',
    yaxis_title='MSE (log)',
    xaxis=dict(
        type='log',  # if r values are logarithmic in spacing
        tickvals=r_values,
        ticktext=[str(r) for r in r_values]
    ),
    yaxis=dict(type='log'),
    template='plotly_white',
    legend_title='Noise Level',
    hovermode='x unified',
    margin=dict(t=30, b=30, l=20, r=20),
)

In [131]:
n_run = 5
r_list = [2, 4, 8, 16, 32, 64]
noise_levels = [10, 20, 30]
solvers = ["ols", "lts"]  # The two solvers to compare
eps = 0.1
# --- 1. Run Experiments for Both Solvers ---
all_results = []  # List to hold dicts: {solver, r, noise, mse_mean, mse_std}

for solver in solvers:
    print(f"Running solver: {solver}...")
    for r in r_list:
        for ns in noise_levels:
            mse_trials = []
            for _ in range(n_run):
                # Call denoising process with current solver
                X_denoised, _ = denoising_process(
                    centers, r, ns, eps, scheme="dense", solver=solver
                )
                mse = np.sum((X_denoised - centers) ** 2)
                mse_trials.append(mse)
            
            all_results.append({
                'solver': solver,
                'r': r,
                'noise': ns,
                'mse_mean': np.mean(mse_trials),
                'mse_std': np.std(mse_trials, ddof=1)
            })


Running solver: ols...
Running solver: lts...


100%|██████████| 10/10 [00:00<00:00, 47.13it/s]


In [134]:

# --- 2. Plotting Comparison ---

# Option A: Overlay Plot (Recommended)
fig = go.Figure()

# Define visual mapping for solvers
style_map = {
    "ols": {"color": None, "dash": "solid", "marker": "circle"},
    "lts": {"color": None, "dash": "dash", "marker": "square"}
}

# Get unique noise levels to generate a color palette
# colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Standard Plotly colors
colors =  ['#636EFA', '#EF553B', '#00CC96']
noise_color_map = {ns: colors[i] for i, ns in enumerate(sorted(noise_levels))}

for solver in solvers:
    for ns in sorted(noise_levels):
        # Filter data for this solver and noise level
        subset = [r for r in all_results if r['solver'] == solver and r['noise'] == ns]
        subset.sort(key=lambda x: x['r'])
        
        x = [res['r'] for res in subset]
        y = [res['mse_mean'] for res in subset]
        y_std = [res['mse_std'] for res in subset]
        
        # Show error bars only for one solver to avoid clutter, or both with opacity
        show_error = True  # Show OLS error bars as reference
        
        fig.add_trace(go.Scatter(
            x=x,
            y=y,
            mode='lines+markers',
            name=f'{solver.upper()} (Noise={ns})',
            line=dict(
                color=noise_color_map[ns],
                dash=style_map[solver]["dash"],
                width=2
            ),
            marker=dict(
                symbol=style_map[solver]["marker"],
                size=8,
                color=noise_color_map[ns]
            ),
            error_y=dict(
                type='data',
                array=y_std,
                visible=show_error,
                thickness=1.5,
                width=6
            ) if show_error else None,
            legendgroup=f'noise_{ns}',
            legendgrouptitle_text=f'Noise = {ns}'
        ))

fig.update_layout(
    # title='MSE Comparison: OLS (solid) vs LTS (dashed)',
    xaxis_title='r (log)',
    yaxis_title='MSE (log)',
    yaxis=dict(type='log'),
    xaxis=dict(type='log', tickvals=r_list, ticktext=[str(r) for r in r_list]),
    template='plotly_white',
    hovermode='x unified',
    legend=dict(groupclick="toggleitem"),  # Allows toggling individual traces
    margin=dict(t=30, b=30, l=20, r=20),
)

fig.show()

# Non iid partition

In [139]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Parameters
n_classes = 15
n_clients = 10
alphas = [0.1, 0.5, 1] # Three levels: highly non-IID to nearly IID

# Create subplots: 1 row, 3 columns
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=[f'α = {a}' for a in alphas],
    horizontal_spacing=0.05
)

for i, alpha in enumerate(alphas):
    # 1. Partition Data (Assumes your dirichlet_data_partition function is defined)
    X_splits, y_splits = dirichlet_data_partition(X, y, n_client=n_clients, alpha=alpha, random_state=42)
    
    # 2. Build Count Matrix
    count_matrix = np.zeros((n_clients, n_classes), dtype=int)
    for client_id, y_client in enumerate(y_splits):
        # Using minlength to ensure all classes are represented
        class_counts = np.bincount(y_client.astype(int), minlength=n_classes+1)
        count_matrix[client_id] = class_counts[1:16]
    
    # 3. Add Heatmap Trace
    fig.add_trace(
        go.Heatmap(
            z=count_matrix,
            x=[f'Class {j}' for j in range(n_classes)],
            y=[f'Client {j}' for j in range(n_clients)],
            coloraxis="coloraxis", # Shared color scale across all subplots
            hovertemplate='Client: %{y}<br>Class: %{x}<br>Count: %{z}<extra></extra>'
        ),
        row=1, col=i+1
    )

# 4. Update Layout for a cleaner look
fig.update_layout(
    height=450,
    width=1200,
    # title_text='Effect of Dirichlet α on Label Distribution',
    coloraxis=dict(colorscale='YlGnBu', colorbar_title='Samples'),
    showlegend=False
)

# Tighten layout and adjust vertical size
fig.update_layout(margin=dict(t=80, b=50, l=50, r=50))

fig.show()

In [ ]:
summary_s1 = pd.read_pickle("results/dirichlet_ess_S1_20260430_160734.pkl")['summary']
summary_s2 = pd.read_pickle("results/dirichlet_ess_S2_20260430_160927.pkl")['summary']
summary_s3 = pd.read_pickle("results/dirichlet_ess_S3_20260430_161042.pkl")['summary']
summary_s4 = pd.read_pickle("results/dirichlet_ess_S4_20260430_161305.pkl")['summary']
all_summaries = pd.concat({'s1': summary_s1, 's2': summary_s2, 's3': summary_s3, 's4': summary_s4}, axis=0)
all_summaries = all_summaries.melt(ignore_index=False).reset_index()

In [31]:
all_summaries.to_clipboard()

In [45]:
df = all_summaries
# ── pivot to wide format ─────────────────────────────────────────────────────
df_mean = df[df['variable_1'] == 'mean'].pivot_table(
    index=['level_0', 'experiment'], columns='variable_0', values='value'
).reset_index()

df_std = df[df['variable_1'] == 'std'].pivot_table(
    index=['level_0', 'experiment'], columns='variable_0', values='value'
).reset_index()

df_wide = df_mean.merge(df_std, on=['level_0', 'experiment'],
                        suffixes=('_mean', '_std'))

# ── parse experiment labels ──────────────────────────────────────────────────
def parse_experiment(exp):
    if exp == 'Centralized':
        return 'Centralized', None
    parts = exp.rsplit(' a=', 1)
    return parts[0].strip(), parts[1].strip()

df_wide[['scheme', 'alpha']] = pd.DataFrame(
    df_wide['experiment'].apply(parse_experiment).tolist(),
    index=df_wide.index
)

# ── config ───────────────────────────────────────────────────────────────────
datasets       = ['s1', 's2', 's3', 's4']
dataset_labels = ['S1', 'S2', 'S3', 'S4']
schemes        = ['ESS', 'Uniform', 'Cluster Size']
alphas         = ['0.5', '1', 'inf']
alpha_labels   = ['α=0.5', 'α=1', 'α=∞']
metric         = 'ARI'   # change to 'ARI' or 'Purity' as needed

colors   = {'ESS': '#636EFA', 'Uniform': '#EF553B', 'Cluster Size': '#00CC96'}
patterns = {'ESS': '', 'Uniform': '/', 'Cluster Size': 'x'}

# ── build figure ─────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f'Dataset {l}' for l in dataset_labels],
    shared_yaxes=True,
    horizontal_spacing=0.04
)

show_legend = True   # only add legend entries once

for idx, (ds, ds_label) in enumerate(zip(datasets, dataset_labels)):
    col = idx + 1

    sub = df_wide[df_wide['level_0'] == ds]

    # ── bars for each weighting scheme ──────────────────────────────────
    for scheme in schemes:
        s = sub[sub['scheme'] == scheme].copy()
        s['alpha'] = pd.Categorical(s['alpha'], categories=alphas, ordered=True)
        s = s.sort_values('alpha')

        fig.add_trace(go.Bar(
            name=scheme,
            x=alpha_labels,
            y=s[f'{metric}_mean'].values,
            error_y=dict(type='data', array=s[f'{metric}_std'].values,
                         visible=True, thickness=1.5, width=4),
            marker_color=colors[scheme],
            legendgroup=scheme,
            showlegend=show_legend,
            offsetgroup=scheme,
        ), row=1, col=col)

    # ── centralised oracle line + shaded band ────────────────────────────
    c = sub[sub['scheme'] == 'Centralized'].iloc[0]
    c_mean = c[f'{metric}_mean']
    c_std  = c[f'{metric}_std']

    # shaded band
    fig.add_trace(go.Scatter(
        x=alpha_labels + alpha_labels[::-1],
        y=[c_mean + c_std] * 3 + [c_mean - c_std] * 3,
        fill='toself',
        fillcolor='rgba(220,50,50,0.12)',
        line=dict(color='rgba(255,255,255,0)'),
        legendgroup='Centralized',
        showlegend=False,
        hoverinfo='skip',
    ), row=1, col=col)

    # dashed line
    fig.add_trace(go.Scatter(
        x=alpha_labels,
        y=[c_mean] * 3,
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name='Centralized',
        legendgroup='Centralized',
        showlegend=show_legend,
    ), row=1, col=col)

    show_legend = False   # only first subplot contributes legend

    # ── axis labels ──────────────────────────────────────────────────────
    fig.update_xaxes(
        title_text='α<sub>Dir</sub>',
        row=1, col=col
    )

# ── shared y-axis label and range ────────────────────────────────────────────
fig.update_yaxes(
    title_text=metric,
    range=[0.4, 1.0],
    row=1, col=1
)
fig.update_yaxes(
    range=[0.4, 1.0],
    row=1, col=2
)
fig.update_yaxes(
    range=[0.4, 1.0],
    row=1, col=3
)
fig.update_yaxes(
    range=[0.4, 1.0],
    row=1, col=4
)

# ── global layout ─────────────────────────────────────────────────────────────
fig.update_layout(
    # title=dict(
    #     text='Effect of ESS Weighting under Non-IID Data Partition',
    #     x=0.5, font=dict(size=15)
    # ),
    barmode='group',
    height=400, width=1600,
    legend=dict(
        orientation='h',
        yanchor='bottom', y=-0.30,
        xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
        # bordercolor='lightgrey', borderwidth=1
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(size=12),
)

fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=True, gridcolor='lightgrey', gridwidth=0.5)
fig.update_layout(margin=dict(t=50, b=50, l=50, r=50))
fig.show()

# Perf vs r 

In [59]:
summary_s1 = pd.read_pickle("results/perf_r_awgn_S1_20260430_201644.pkl")['summary']
summary_s2 = pd.read_pickle("results/perf_r_awgn_S2_20260430_201609.pkl")['summary']
summary_s3 = pd.read_pickle("results/perf_r_awgn_S3_20260430_201541.pkl")['summary']
summary_s4 = pd.read_pickle("results/perf_r_awgn_S4_20260430_201343.pkl")['summary']
all_summaries = pd.concat({'s1': summary_s1, 's2': summary_s2, 's3': summary_s3, 's4': summary_s4}, axis=0)
all_summaries = all_summaries.melt(ignore_index=False).reset_index()

In [76]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── assume df is already loaded ───────────────────────────────────────────────
df = all_summaries.rename(columns={
    "level_0":    "dataset",
    "variable_0": "metric",
    "variable_1": "stat",
})

df_wide = df.pivot_table(
    index=["dataset", "experiment", "metric"],
    columns="stat",
    values="value"
).reset_index()

# ── constants ─────────────────────────────────────────────────────────────────
r_order   = ["r=1", "r=2", "r=4", "r=8", "r=16", "r=32", "r=64"]
r_values  = [1, 2, 4, 8, 16, 32, 64]
datasets  = ["s1", "s2", "s3", "s4"]
metrics   = ["NMI", "ARI", "Purity"]

PALETTE = {
    "NMI":    "#636EFA",
    "ARI":    "#EF553B",
    "Purity": "#00CC96",
}

# ── subplots: 1 row × 4 cols, y-axes truly shared ────────────────────────────
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f"Dataset {d.upper()}" for d in datasets],
    shared_yaxes=True,           # link scales, zoom, pan
)

show_legend = True

for col_idx, ds in enumerate(datasets, start=1):
    ds_df = df_wide[df_wide["dataset"] == ds]

    for metric in metrics:
        met_df = ds_df[ds_df["metric"] == metric]

        tr_df = (
            met_df[met_df["experiment"].isin(r_order)]
            .copy()
        )
        tr_df["r_val"] = tr_df["experiment"].str.replace("r=", "").astype(int)
        tr_df = tr_df.sort_values("r_val")

        fig.add_trace(
            go.Scatter(
                x=tr_df["r_val"],
                y=tr_df["mean"],
                error_y=dict(
                    type="data",
                    array=tr_df["std"].values,
                    visible=True,
                    thickness=1.2,
                    width=4,
                ),
                mode="lines+markers",
                name=metric,
                legendgroup=metric,
                showlegend=show_legend,
                line=dict(color=PALETTE[metric], width=2),
                marker=dict(size=6, color=PALETTE[metric]),
            ),
            row=1, col=col_idx,
        )

    show_legend = False   # legend entries only from first subplot

# ── x‑axes: always visible, log scale ────────────────────────────────────────
for col_idx in range(1, 5):
    fig.update_xaxes(
        type="log",
        tickvals=r_values,
        ticktext=[str(r) for r in r_values],
        title_text="r (log)",
        row=1, col=col_idx,
        showgrid=True,
        gridcolor="#EEEEEE",
        linecolor="#CCCCCC",
    )
# Grid on all axes – horizontal and vertical lines
for col_idx in range(1, 5):
    fig.update_xaxes(
        showgrid=True, gridcolor="#EEEEEE",  # vertical grid lines at x ticks
        row=1, col=col_idx,
    )
    fig.update_yaxes(
        showgrid=True, gridcolor="#EEEEEE",  # horizontal grid lines at y ticks
        row=1, col=col_idx,
    )
# ── y‑axes: shared, but only the first one shows title & tick labels ─────────
fig.update_yaxes(
    title_text="Score",         # shown on first subplot
    showticklabels=True,        # tick labels visible on first subplot
    row=1, col=1,
)

# For subsequent subplots, hide the y‑axis tick labels and title (default when
# shared_yaxes=True, but we make it explicit)
for col_idx in range(2, 5):
    fig.update_yaxes(
        showticklabels=False,   # hide numbers
        title_text=None,        # no “Score” text
        row=1, col=col_idx,
    )

# Apply the common y‑range once – all share the same axis
fig.update_yaxes(range=[0.4, 1.0], row=1, col=1)

# Style
fig.update_yaxes(
    showgrid=True,
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row='all', col='all',
)

# ── layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    width=1600,
    height=400,
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation='h',
        yanchor='bottom', y=-0.30,
        xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
    ),
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.show()

# Bench

In [116]:
summary_s1 = pd.read_pickle("bench_S1.pkl")['summary']
summary_s2 = pd.read_pickle("bench_S2.pkl")['summary']
summary_s3 = pd.read_pickle("bench_S3.pkl")['summary']
summary_s4 = pd.read_pickle("bench_S4.pkl")['summary']
all_summaries = pd.concat({'S1': summary_s1, 'S2': summary_s2, 'S3': summary_s3, 'S4': summary_s4}, axis=0)
all_summaries = all_summaries.melt(ignore_index=False).reset_index()

In [117]:
import pandas as pd

# ── assume df already loaded ──────────────────────────────────────────────────
df = all_summaries.rename(columns={
    "level_0":    "dataset",
    "variable_0": "metric",
    "variable_1": "stat",
})

# ── keep only ARI, NMI, Purity ────────────────────────────────────────────────
df = df[df["metric"].isin(["ARI", "NMI", "Purity"])]

# ── pivot wide ────────────────────────────────────────────────────────────────
df_wide = df.pivot_table(
    index=["dataset", "experiment", "metric"],
    columns="stat",
    values="value"
).reset_index()

# ── format mean ± std as string ───────────────────────────────────────────────
df_wide["value_str"] = (
    df_wide["mean"].map("{:.3f}".format)
    + " $\\pm$ "
    + df_wide["std"].map("{:.3f}".format)
)

# ── classify rows into method / scenario ──────────────────────────────────────
def classify(exp):
    if exp == "Centralized":
        return "Centralised", "—"
    elif exp.startswith("Naive"):
        scen = exp.split("Scenario ")[-1]
        return "Naive", f"Scenario {scen}"
    else:
        scen = exp.split("Scenario ")[-1]
        return "TR-FedKR", f"Scenario {scen}"

df_wide[["method", "scenario"]] = pd.DataFrame(
    df_wide["experiment"].apply(classify).tolist(),
    index=df_wide.index
)

# ── ordering ──────────────────────────────────────────────────────────────────
methods    = ["Centralised", "TR-FedKR", "Naive"]
metrics    = ["NMI", "ARI", "Purity"]
ds_order   = ["S1", "S2", "S3", "S4"]
scen_order = ["Scenario 1", "Scenario 2", "Scenario 3", "Scenario 4", "Scenario 5"]
n_scen     = len(scen_order)

# ── lookup dictionaries (string values) ──────────────────────────────────────
cent_vals = (
    df_wide[df_wide["method"] == "Centralised"]
    .set_index(["dataset", "metric"])["value_str"]
    .to_dict()
)

fed_vals = (
    df_wide[df_wide["method"] != "Centralised"]
    .set_index(["dataset", "scenario", "method", "metric"])["value_str"]
    .to_dict()
)

# ── lookup dictionary for raw mean values (for comparison) ───────────────────
fed_means = (
    df_wide[df_wide["method"] != "Centralised"]
    .set_index(["dataset", "scenario", "method", "metric"])["mean"]
    .to_dict()
)

# ── helper: determine which method wins per (dataset, scenario, metric) ───────
def get_winner(ds, sc, met):
    """Return the method with the highest mean for this (ds, sc, met) pair."""
    tr_mean    = fed_means.get((ds, sc, "TR-FedKR", met), None)
    naive_mean = fed_means.get((ds, sc, "Naive",    met), None)

    if tr_mean is None and naive_mean is None:
        return None
    if tr_mean is None:
        return "Naive"
    if naive_mean is None:
        return "TR-FedKR"
    if tr_mean > naive_mean:
        return "TR-FedKR"
    elif naive_mean > tr_mean:
        return "Naive"
    else:
        return "both"   # exact tie → bold both

def maybe_bold(val_str, meth, ds, sc, met):
    """Wrap val_str in \textbf{} if this method is the winner."""
    winner = get_winner(ds, sc, met)
    if winner is None:
        return val_str
    if winner == "both" or winner == meth:
        return f"\\textbf{{{val_str}}}"
    return val_str

# ── column spec: centred Dataset and Scenario columns ────────────────────────
n_data_cols = len(methods) * len(metrics)
col_spec = (
    "{>{\\centering\\arraybackslash}l"
    "@{\\hspace{0.8em}}"
    ">{\\centering\\arraybackslash}l"
    + "l" * n_data_cols
    + "}"
)

# ── header row 1: method group spans ─────────────────────────────────────────
header_line1 = " & & " + " & ".join(
    f"\\multicolumn{{{len(metrics)}}}{{c}}{{{m}}}" for m in methods
) + " \\\\"

# ── cmidrules for each method group (columns start at 3) ─────────────────────
cmidrules = ""
for i, m in enumerate(methods):
    start = 3 + i * len(metrics)
    end   = start + len(metrics) - 1
    cmidrules += f"\\cmidrule(lr){{{start}-{end}}} "

# ── header row 2: column labels ───────────────────────────────────────────────
header_line2 = "Dataset & Scenario & " + " & ".join(
    metrics * len(methods)
) + " \\\\"

# ── body rows ─────────────────────────────────────────────────────────────────
body_lines = []
for ds in ds_order:
    for j, sc in enumerate(scen_order):
        row_parts = []

        # ── Dataset column: multirow spanning all scenarios ───────────────
        if j == 0:
            row_parts.append(f"\\multirow{{{n_scen}}}{{*}}{{{ds}}}")
        else:
            row_parts.append("")

        # ── Scenario column ───────────────────────────────────────────────
        row_parts.append(sc)

        # ── Centralised columns: multirow on first row, empty on others ──
        # (no bolding for Centralised — it is the reference)
        for met in metrics:
            val = cent_vals.get((ds, met), "—")
            if j == 0:
                row_parts.append(
                    f"\\multirow{{{n_scen}}}{{*}}{{{val}}}"
                )
            else:
                row_parts.append("")

        # ── TR-FedKR and Naive columns (with bolding) ─────────────────────
        for meth in ["TR-FedKR", "Naive"]:
            for met in metrics:
                val = fed_vals.get((ds, sc, meth, met), "—")
                val = maybe_bold(val, meth, ds, sc, met)
                row_parts.append(val)

        body_lines.append(" & ".join(row_parts) + " \\\\")

    # ── midrule between dataset blocks ───────────────────────────────────
    if ds != ds_order[-1]:
        body_lines.append("\\midrule")

# ── caption ───────────────────────────────────────────────────────────────────
caption = (
    "Clustering performance (mean $\\pm$ std over 20 trials) across "
    "datasets S1--S4 and four noise scenarios. "
    "Bold indicates the best federated method per row."
)

# ── assemble full LaTeX ───────────────────────────────────────────────────────
latex = f"""\
\\begin{{table*}}[htbp]
\\centering
\\caption{{{caption}}}
\\label{{tab:main_results}}
\\resizebox{{\\textwidth}}{{!}}{{%
\\begin{{tabular}}{col_spec}
\\toprule
{header_line1}
{cmidrules.strip()}
{header_line2}
\\midrule
{chr(10).join(body_lines)}
\\bottomrule
\\end{{tabular}}%
}}
\\end{{table*}}"""

print(latex)


\begin{table*}[htbp]
\centering
\caption{Clustering performance (mean $\pm$ std over 20 trials) across datasets S1--S4 and four noise scenarios. Bold indicates the best federated method per row.}
\label{tab:main_results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{>{\centering\arraybackslash}l@{\hspace{0.8em}}>{\centering\arraybackslash}llllllllll}
\toprule
 & & \multicolumn{3}{c}{Centralised} & \multicolumn{3}{c}{TR-FedKR} & \multicolumn{3}{c}{Naive} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}
Dataset & Scenario & NMI & ARI & Purity & NMI & ARI & Purity & NMI & ARI & Purity \\
\midrule
\multirow{5}{*}{S1} & Scenario 1 & \multirow{5}{*}{0.985 $\pm$ 0.006} & \multirow{5}{*}{0.982 $\pm$ 0.018} & \multirow{5}{*}{0.990 $\pm$ 0.014} & \textbf{0.977 $\pm$ 0.014} & \textbf{0.962 $\pm$ 0.042} & \textbf{0.973 $\pm$ 0.037} & 0.936 $\pm$ 0.022 & 0.867 $\pm$ 0.051 & 0.889 $\pm$ 0.050 \\
 & Scenario 2 &  &  &  & \textbf{0.968 $\pm$ 0.018} & \textbf{0.940 $\pm$ 0.050} & \textbf{0.95

# REal world

In [129]:
statlog = pd.read_pickle("bench_statlog.pkl")['summary']
letter = pd.read_pickle("bench_letter.pkl")['summary']
wine = pd.read_pickle("bench_wine.pkl")['summary']
yeast = pd.read_pickle("bench_yeast.pkl")['summary']
all_summaries = pd.concat({'Statlog': statlog, 'Letter': letter, 'Wine': wine, 'Yeast': yeast}, axis=0)
all_summaries = all_summaries.melt(ignore_index=False).reset_index()

In [130]:
import pandas as pd

# ── assume df already loaded ──────────────────────────────────────────────────
df = all_summaries.rename(columns={
    "level_0":    "dataset",
    "variable_0": "metric",
    "variable_1": "stat",
})

# ── keep only ARI, NMI, Purity ────────────────────────────────────────────────
df = df[df["metric"].isin(["ARI", "NMI", "Purity"])]

# ── pivot wide ────────────────────────────────────────────────────────────────
df_wide = df.pivot_table(
    index=["dataset", "experiment", "metric"],
    columns="stat",
    values="value"
).reset_index()

# ── format mean ± std as string ───────────────────────────────────────────────
df_wide["value_str"] = (
    df_wide["mean"].map("{:.3f}".format)
    + " $\\pm$ "
    + df_wide["std"].map("{:.3f}".format)
)

# ── classify rows into method / scenario ──────────────────────────────────────
def classify(exp):
    if exp == "Centralized":
        return "Centralised", "—"
    elif exp.startswith("Naive"):
        scen = exp.split("Scenario ")[-1]
        return "Naive", f"Scenario {scen}"
    else:
        scen = exp.split("Scenario ")[-1]
        return "TR-FedKR", f"Scenario {scen}"

df_wide[["method", "scenario"]] = pd.DataFrame(
    df_wide["experiment"].apply(classify).tolist(),
    index=df_wide.index
)

# ── ordering ──────────────────────────────────────────────────────────────────
methods    = ["Centralised", "TR-FedKR", "Naive"]
metrics    = ["NMI", "ARI", "Purity"]
ds_order   = [ "Wine", "Yeast", "Statlog", "Letter"]
scen_order = ["Scenario 1", "Scenario 2", "Scenario 3", "Scenario 4", "Scenario 5"]
n_scen     = len(scen_order)

# ── lookup dictionaries (string values) ──────────────────────────────────────
cent_vals = (
    df_wide[df_wide["method"] == "Centralised"]
    .set_index(["dataset", "metric"])["value_str"]
    .to_dict()
)
# print(cent_vals)
fed_vals = (
    df_wide[df_wide["method"] != "Centralised"]
    .set_index(["dataset", "scenario", "method", "metric"])["value_str"]
    .to_dict()
)

# ── lookup dictionary for raw mean values (for comparison) ───────────────────
fed_means = (
    df_wide[df_wide["method"] != "Centralised"]
    .set_index(["dataset", "scenario", "method", "metric"])["mean"]
    .to_dict()
)

# ── helper: determine which method wins per (dataset, scenario, metric) ───────
def get_winner(ds, sc, met):
    """Return the method with the highest mean for this (ds, sc, met) pair."""
    tr_mean    = fed_means.get((ds, sc, "TR-FedKR", met), None)
    naive_mean = fed_means.get((ds, sc, "Naive",    met), None)

    if tr_mean is None and naive_mean is None:
        return None
    if tr_mean is None:
        return "Naive"
    if naive_mean is None:
        return "TR-FedKR"
    if tr_mean > naive_mean:
        return "TR-FedKR"
    elif naive_mean > tr_mean:
        return "Naive"
    else:
        return "both"   # exact tie → bold both

def maybe_bold(val_str, meth, ds, sc, met):
    """Wrap val_str in \textbf{} if this method is the winner."""
    winner = get_winner(ds, sc, met)
    if winner is None:
        return val_str
    if winner == "both" or winner == meth:
        return f"\\textbf{{{val_str}}}"
    return val_str

# ── column spec: centred Dataset and Scenario columns ────────────────────────
n_data_cols = len(methods) * len(metrics)
col_spec = (
    "{>{\\centering\\arraybackslash}l"
    "@{\\hspace{0.8em}}"
    ">{\\centering\\arraybackslash}l"
    + "l" * n_data_cols
    + "}"
)

# ── header row 1: method group spans ─────────────────────────────────────────
header_line1 = " & & " + " & ".join(
    f"\\multicolumn{{{len(metrics)}}}{{c}}{{{m}}}" for m in methods
) + " \\\\"

# ── cmidrules for each method group (columns start at 3) ─────────────────────
cmidrules = ""
for i, m in enumerate(methods):
    start = 3 + i * len(metrics)
    end   = start + len(metrics) - 1
    cmidrules += f"\\cmidrule(lr){{{start}-{end}}} "

# ── header row 2: column labels ───────────────────────────────────────────────
header_line2 = "Dataset & Scenario & " + " & ".join(
    metrics * len(methods)
) + " \\\\"

# ── body rows ─────────────────────────────────────────────────────────────────
body_lines = []
for ds in ds_order:
    for j, sc in enumerate(scen_order):
        row_parts = []

        # ── Dataset column: multirow spanning all scenarios ───────────────
        if j == 0:
            row_parts.append(f"\\multirow{{{n_scen}}}{{*}}{{{ds}}}")
        else:
            row_parts.append("")

        # ── Scenario column ───────────────────────────────────────────────
        row_parts.append(sc)

        # ── Centralised columns: multirow on first row, empty on others ──
        # (no bolding for Centralised — it is the reference)
        for met in metrics:
            val = cent_vals.get((ds, met), "—")
            if j == 0:
                row_parts.append(
                    f"\\multirow{{{n_scen}}}{{*}}{{{val}}}"
                )
            else:
                row_parts.append("")

        # ── TR-FedKR and Naive columns (with bolding) ─────────────────────
        for meth in ["TR-FedKR", "Naive"]:
            for met in metrics:
                val = fed_vals.get((ds, sc, meth, met), "—")
                val = maybe_bold(val, meth, ds, sc, met)
                row_parts.append(val)

        body_lines.append(" & ".join(row_parts) + " \\\\")

    # ── midrule between dataset blocks ───────────────────────────────────
    if ds != ds_order[-1]:
        body_lines.append("\\midrule")

# ── caption ───────────────────────────────────────────────────────────────────
caption = (
    "Clustering performance (mean $\\pm$ std over 20 trials) across "
    "four datasets and three noise scenarios. "
    "Bold indicates the best federated method per row."
)

# ── assemble full LaTeX ───────────────────────────────────────────────────────
latex = f"""\
\\begin{{table*}}[htbp]
\\centering
\\caption{{{caption}}}
\\label{{tab:main_results}}
\\resizebox{{\\textwidth}}{{!}}{{%
\\begin{{tabular}}{col_spec}
\\toprule
{header_line1}
{cmidrules.strip()}
{header_line2}
\\midrule
{chr(10).join(body_lines)}
\\bottomrule
\\end{{tabular}}%
}}
\\end{{table*}}"""

print(latex)


\begin{table*}[htbp]
\centering
\caption{Clustering performance (mean $\pm$ std over 20 trials) across four datasets and three noise scenarios. Bold indicates the best federated method per row.}
\label{tab:main_results}
\resizebox{\textwidth}{!}{%
\begin{tabular}{>{\centering\arraybackslash}l@{\hspace{0.8em}}>{\centering\arraybackslash}llllllllll}
\toprule
 & & \multicolumn{3}{c}{Centralised} & \multicolumn{3}{c}{TR-FedKR} & \multicolumn{3}{c}{Naive} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8} \cmidrule(lr){9-11}
Dataset & Scenario & NMI & ARI & Purity & NMI & ARI & Purity & NMI & ARI & Purity \\
\midrule
\multirow{5}{*}{Wine} & Scenario 1 & \multirow{5}{*}{0.029 $\pm$ 0.001} & \multirow{5}{*}{0.013 $\pm$ 0.001} & \multirow{5}{*}{0.457 $\pm$ 0.001} & \textbf{0.029 $\pm$ 0.001} & \textbf{0.014 $\pm$ 0.001} & 0.456 $\pm$ 0.001 & 0.029 $\pm$ 0.001 & 0.013 $\pm$ 0.001 & \textbf{0.456 $\pm$ 0.001} \\
 & Scenario 2 &  &  &  & \textbf{0.029 $\pm$ 0.001} & \textbf{0.013 $\pm$ 0.001} & \textbf{0.4

# Scaling

In [131]:
df = pd.read_pickle("scaling_letter.pkl")['summary']
df = df.melt(ignore_index=False).reset_index()


In [134]:
df.to_clipboard()

In [138]:
import plotly.graph_objects as go
import pandas as pd

# ── assume df is already loaded ───────────────────────────────────────────────
# df has columns: experiment, variable_0, variable_1, value

df_wide = df.pivot_table(
    index=["experiment", "variable_0"],
    columns="variable_1",
    values="value"
).reset_index().rename(columns={"variable_0": "metric"})

# ── constants ─────────────────────────────────────────────────────────────────
n_order  = ["N = 8", "N = 16", "N = 32", "N = 64", "N = 128", "N = 256"]
n_values = [8, 16, 32, 64, 128, 256]
metrics  = ["NMI", "ARI", "Purity"]

PALETTE = {
    "NMI":    "#636EFA",
    "ARI":    "#EF553B",
    "Purity": "#00CC96",
}

# ── single figure ─────────────────────────────────────────────────────────────
fig = go.Figure()

for metric in metrics:
    met_df = df_wide[df_wide["metric"] == metric].copy()

    met_df = (
        met_df[met_df["experiment"].isin(n_order)]
        .copy()
    )
    met_df["n_val"] = met_df["experiment"].str.replace("N = ", "").astype(int)
    met_df = met_df.sort_values("n_val")

    fig.add_trace(
        go.Scatter(
            x=met_df["n_val"],
            y=met_df["mean"],
            error_y=dict(
                type="data",
                array=met_df["std"].values,
                visible=True,
                thickness=1.2,
                width=4,
            ),
            mode="lines+markers",
            name=metric,
            legendgroup=metric,
            line=dict(color=PALETTE[metric], width=2),
            marker=dict(size=6, color=PALETTE[metric]),
        )
    )

# ── x-axis: log scale ─────────────────────────────────────────────────────────
fig.update_xaxes(
    type="log",
    tickvals=n_values,
    ticktext=[str(n) for n in n_values],
    title_text="N (log)",
    showgrid=True,
    gridcolor="#EEEEEE",
    # linecolor="#CCCCCC",
)

# ── y-axis ────────────────────────────────────────────────────────────────────
fig.update_yaxes(
    title_text="Score",
    showgrid=True,
    gridcolor="#EEEEEE",
    # linecolor="#CCCCCC",
    showticklabels=True,
)

# ── layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation="h",
        yanchor="bottom", y=-0.25,
        xanchor="center", x=0.5,
        bgcolor="rgba(255,255,255,0.9)",
    ),
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.show()


# k' sensitivity

In [139]:
summary_s1 = pd.read_pickle("perf_k_S1.pkl")['summary']
summary_s2 = pd.read_pickle("perf_k_S2.pkl")['summary']
summary_s3 = pd.read_pickle("perf_k_S3.pkl")['summary']
summary_s4 = pd.read_pickle("perf_k_S4.pkl")['summary']
all_summaries = pd.concat({'s1': summary_s1, 's2': summary_s2, 's3': summary_s3, 's4': summary_s4}, axis=0)
all_summaries = all_summaries.melt(ignore_index=False).reset_index()

In [144]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── assume df is already loaded ───────────────────────────────────────────────
df = all_summaries.rename(columns={
    "level_0":    "dataset",
    "variable_0": "metric",
    "variable_1": "stat",
})

df_wide = df.pivot_table(
    index=["dataset", "experiment", "metric"],
    columns="stat",
    values="value"
).reset_index()

# ── constants ─────────────────────────────────────────────────────────────────
k_order  = ["k'=5", "k'=10", "k'=15", "k'=20", "k'=25", "k'=30", "k'=35", "k'=40"]
k_values = [5, 10, 15, 20, 25, 30, 35, 40]
datasets = ["s1", "s2", "s3", "s4"]
metrics  = ["NMI", "ARI", "Purity"]

PALETTE = {
    "NMI":    "#636EFA",
    "ARI":    "#EF553B",
    "Purity": "#00CC96",
}

# ── subplots: 1 row × 4 cols, y-axes truly shared ────────────────────────────
fig = make_subplots(
    rows=1, cols=4,
    subplot_titles=[f"Dataset {d.upper()}" for d in datasets],
    shared_yaxes=True,           # link scales, zoom, pan
)

show_legend = True

for col_idx, ds in enumerate(datasets, start=1):
    ds_df = df_wide[df_wide["dataset"] == ds]

    for metric in metrics:
        met_df = ds_df[ds_df["metric"] == metric]

        tr_df = (
            met_df[met_df["experiment"].isin(k_order)]
            .copy()
        )
        tr_df["k_val"] = tr_df["experiment"].str.replace("k'=", "").astype(int)
        tr_df = tr_df.sort_values("k_val")

        fig.add_trace(
            go.Scatter(
                x=tr_df["k_val"],
                y=tr_df["mean"],
                error_y=dict(
                    type="data",
                    array=tr_df["std"].values,
                    visible=True,
                    thickness=1.2,
                    width=4,
                ),
                mode="lines+markers",
                name=metric,
                legendgroup=metric,
                showlegend=show_legend,
                line=dict(color=PALETTE[metric], width=2),
                marker=dict(size=6, color=PALETTE[metric]),
            ),
            row=1, col=col_idx,
        )

    show_legend = False   # legend entries only from first subplot

# ── x‑axes: always visible ───────────────────────────────────────────────────
for col_idx in range(1, 5):
    fig.update_xaxes(
        tickvals=k_values,
        ticktext=[str(k) for k in k_values],
        title_text="k'",
        row=1, col=col_idx,
        showgrid=True,
        gridcolor="#EEEEEE",
        linecolor="#CCCCCC",
    )

# Grid on all axes – horizontal and vertical lines
for col_idx in range(1, 5):
    fig.update_xaxes(
        showgrid=True, gridcolor="#EEEEEE",  # vertical grid lines at x ticks
        row=1, col=col_idx,
    )
    fig.update_yaxes(
        showgrid=True, gridcolor="#EEEEEE",  # horizontal grid lines at y ticks
        row=1, col=col_idx,
    )

# ── y‑axes: shared, but only the first one shows title & tick labels ─────────
fig.update_yaxes(
    title_text="Score",         # shown on first subplot
    showticklabels=True,        # tick labels visible on first subplot
    row=1, col=1,
)

# For subsequent subplots, hide the y‑axis tick labels and title (default when
# shared_yaxes=True, but we make it explicit)
for col_idx in range(2, 5):
    fig.update_yaxes(
        showticklabels=False,   # hide numbers
        title_text=None,        # no "Score" text
        row=1, col=col_idx,
    )

# Apply the common y‑range once – all share the same axis
fig.update_yaxes(range=[0.4, 1.0], row=1, col=1)

# Style
fig.update_yaxes(
    showgrid=True,
    gridcolor="#EEEEEE",
    linecolor="#CCCCCC",
    row='all', col='all',
)

# ── layout ────────────────────────────────────────────────────────────────────
fig.update_layout(
    width=1600,
    height=400,
    plot_bgcolor="white",
    paper_bgcolor="white",
    legend=dict(
        orientation='h',
        yanchor='bottom', y=-0.30,
        xanchor='center', x=0.5,
        bgcolor='rgba(255,255,255,0.9)',
    ),
    margin=dict(t=50, b=50, l=50, r=50),
)

fig.show()
